# Award recipients, phase 1: the countries the raw tables already hold

Builds `openalex.awards.award_recipients_raw`, the side table `nightly_grants/lib/award_recipients.py` reads (oxjob #1570,
funding-flows work under the Wellcome funding-metadata grant 323416/Z/24/Z). One row per award and country or region, keyed by the
producer's STAGING triple (staging_id, funder_id, award_key), so rows follow merges and redirects; nothing is written to
`openalex_awards_raw`.

Phase 1 carries what the raw tables already hold and the API dropped:
- the four USAspending feeds (`usaid_`, `cdc_`, `dod_`, `usda_awards_raw`): the recipient organisation's country
  (`recipient_country_code`, basis organisation_location) and the place of performance
  (`primary_place_of_performance_country_code`, basis place_of_performance), ISO3 -> ISO2 through `openalex.common.countries`;
- `idrc_raw`: every recipient country with its percentage, and every recipient region (DAC codes), basis funder_reported.
The nightly serves the place of performance when both are present. Measured 2026-10-06 on awards_api v1249: 130,548 awards,
9,331 more awards with an African recipient country (USAID 6,605, CDC 1,585, IDRC 923, DoD 211, USDA 7); 30 of 30 hand-checked
placements match the source record. Rerun whenever the raw tables are refreshed (the table is rebuilt whole).

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.award_recipients_raw AS
SELECT abs(xxhash64(concat(b.funder_id, ':', lower(b.funder_award_id)))) % 9000000000 AS staging_id,
       b.funder_id, lower(b.funder_award_id) AS award_key, b.provenance, b.funder_award_id,
       b.kind, b.country_code, b.region_code, b.region_vocabulary, b.basis, sum(b.share) AS share,
       CASE WHEN max(b.share) IS NOT NULL THEN 'funder_reported' END AS share_basis, b.source_field,
       'funder' AS asserted_by, 'af1570-phase1-v1' AS adapter_version, current_timestamp() AS asserted_at
FROM (
WITH iso AS (SELECT upper(alpha_3) a3, upper(id) a2 FROM openalex.common.countries WHERE alpha_3 IS NOT NULL),
us AS (
  SELECT 'usaspending_usaid' provenance, 4320306115L funder_id, award_id_fain fain, recipient_country_code rc, primary_place_of_performance_country_code pc FROM openalex.awards.usaid_awards_raw
  UNION ALL SELECT 'usaspending_cdc', 4320332162L, award_id_fain, recipient_country_code, primary_place_of_performance_country_code FROM openalex.awards.cdc_awards_raw
  UNION ALL SELECT 'usaspending', 4320306078L, award_id_fain, recipient_country_code, primary_place_of_performance_country_code FROM openalex.awards.dod_awards_raw
  UNION ALL SELECT 'usaspending_usda', 4320306114L, award_id_fain, recipient_country_code, primary_place_of_performance_country_code FROM openalex.awards.usda_awards_raw
),
us_rows AS (
  SELECT provenance, funder_id, fain AS funder_award_id, 'country' kind, i.a2 country_code, CAST(NULL AS STRING) region_code, CAST(NULL AS STRING) region_vocabulary,
         'organisation_location' basis, CAST(NULL AS DOUBLE) share, 'recipient_country_code' source_field
  FROM us JOIN iso i ON upper(us.rc) = i.a3 WHERE fain IS NOT NULL
  UNION ALL
  SELECT provenance, funder_id, fain, 'country', i.a2, NULL, NULL, 'place_of_performance', CAST(NULL AS DOUBLE), 'primary_place_of_performance_country_code'
  FROM us JOIN iso i ON upper(us.pc) = i.a3 WHERE fain IS NOT NULL
),
idrc AS (
  SELECT iati_identifier,
         from_json(recipient_countries_json, 'array<struct<code:string,percentage:string>>') cs,
         from_json(recipient_regions_json, 'array<struct<code:string,percentage:string,vocabulary:string>>') rs
  FROM openalex.awards.idrc_raw
),
idrc_rows AS (
  SELECT 'idrc_iati' provenance, 4320319949L funder_id, iati_identifier funder_award_id, 'country' kind, upper(trim(c.code)) country_code, CAST(NULL AS STRING) region_code, CAST(NULL AS STRING) region_vocabulary,
         'funder_reported' basis, try_cast(c.percentage AS DOUBLE) / 100.0 share, 'recipient-country' source_field
  FROM idrc LATERAL VIEW explode(cs) e AS c
  UNION ALL
  SELECT 'idrc_iati', 4320319949L, iati_identifier, 'region', NULL, r.code, coalesce(r.vocabulary, '1'), 'funder_reported', try_cast(r.percentage AS DOUBLE) / 100.0, 'recipient-region'
  FROM idrc LATERAL VIEW explode(rs) e AS r
)
SELECT DISTINCT * FROM us_rows UNION ALL SELECT DISTINCT * FROM idrc_rows
) b
GROUP BY b.funder_id, b.funder_award_id, b.provenance, b.kind, b.country_code, b.region_code, b.region_vocabulary, b.basis, b.source_field;

In [ ]:
%sql
SELECT provenance, kind, basis, count(*) rows, count(DISTINCT staging_id) awards
FROM openalex.awards.award_recipients_raw GROUP BY ALL ORDER BY ALL

In [ ]:
%sql
-- Shape the nightly checks (RECIPIENT_RAW_SHAPE / RECIPIENT_RAW_UNIQUE): nothing may come back.
SELECT * FROM openalex.awards.award_recipients_raw
WHERE NOT coalesce(length(provenance)>0 AND length(funder_award_id)>0 AND staging_id IS NOT NULL AND funder_id IS NOT NULL AND length(award_key)>0
  AND kind IN ('country','region') AND basis IN ('funder_reported','place_of_performance','organisation_location')
  AND asserted_by IN ('funder','openalex','curated') AND (share IS NULL OR share BETWEEN 0 AND 1)
  AND CASE kind WHEN 'country' THEN country_code RLIKE '^[A-Z]{2}$' AND region_code IS NULL
                WHEN 'region' THEN length(region_code)>0 AND length(region_vocabulary)>0 AND country_code IS NULL END, false)
UNION ALL
SELECT * FROM (SELECT *, count(*) OVER (PARTITION BY provenance,staging_id,funder_id,award_key,kind,country_code,region_code,region_vocabulary,basis) dup
  FROM openalex.awards.award_recipients_raw) WHERE dup > 1